# Homework 6 - GAN (WGAN-GP)
使用 WGAN-GP（Wasserstein GAN + Gradient Penalty）生成動漫人臉。

**核心改動（相對於老師 DCGAN baseline）：**
- Discriminator 改為 Critic（去掉最後的 Sigmoid）
- Loss 改為 Wasserstein distance + Gradient Penalty（λ=10）
- n_critic=5（Critic 每更新 5 次，Generator 更新 1 次）
- Optimizer betas 改為 (0.0, 0.9)

## 環境安裝

In [ ]:
!pip install qqdm

## 下載資料集
根據學號末位選擇對應的下載連結（0~9 對應不同 Google Drive 連結）。

In [ ]:
import gdown

# 根據學號末位選擇連結（0~9）
links = [
    'https://drive.google.com/uc?id=1igSc3vOmYD2eLRCa6nFBMkX729hS5n9S',  # 0
    'https://drive.google.com/uc?id=1VuRFiQPohJXQkPcoFxg_Qk4HW5EFwGqV',  # 1
    'https://drive.google.com/uc?id=1RHNmxCz7v7KXXNzNMCO6MKb8YG8OFgWm',  # 2
    'https://drive.google.com/uc?id=1GQzEK0NjWAjzTb2JBJx2y4m5Z5vXjqtE',  # 3
    'https://drive.google.com/uc?id=1Av_UlNKOVt6o7c6rBFjMj9L_9ZOgXiIh',  # 4
    'https://drive.google.com/uc?id=1RHNmxCz7v7KXXNzNMCO6MKb8YG8OFgWm',  # 5
    'https://drive.google.com/uc?id=1GQzEK0NjWAjzTb2JBJx2y4m5Z5vXjqtE',  # 6
    'https://drive.google.com/uc?id=1igSc3vOmYD2eLRCa6nFBMkX729hS5n9S',  # 7
    'https://drive.google.com/uc?id=1VuRFiQPohJXQkPcoFxg_Qk4HW5EFwGqV',  # 8
    'https://drive.google.com/uc?id=1Av_UlNKOVt6o7c6rBFjMj9L_9ZOgXiIh',  # 9
]

# 請將 student_id_last_digit 改成你的學號末位數字（0~9）
student_id_last_digit = 0
gdown.download(links[student_id_last_digit], 'crypko_data.zip', quiet=False)

In [ ]:
import zipfile
with zipfile.ZipFile('crypko_data.zip', 'r') as zf:
    zf.extractall('.')
print('解壓完成')

## Import 套件

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.io as io
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from qqdm import qqdm

## 固定隨機種子（確保結果可重現）

In [ ]:
def same_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(2021)

## Dataset

In [ ]:
class CrypkoDataset(Dataset):
    def __init__(self, fnames, transform):
        self.transform = transform
        self.fnames = fnames
        self.num_samples = len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = io.read_image(fname)  # 讀取圖片，shape: (C, H, W)
        img = self.transform(img)
        return img

    def __len__(self):
        return self.num_samples


def get_dataset(root):
    fnames = sorted([
        os.path.join(root, x)
        for x in os.listdir(root)
        if x.endswith('.jpg')
    ])
    # 圖片預處理：縮放 → PIL → Tensor → 正規化到 [-1, 1]
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])
    dataset = CrypkoDataset(fnames, transform)
    return dataset


dataset = get_dataset('faces')
print(f'資料集大小：{len(dataset)} 張圖片')

## 模型架構

### 權重初始化

In [ ]:
def weights_init(m):
    """DCGAN 論文建議的初始化方式：Conv 用 N(0, 0.02)，BatchNorm 用 N(1, 0.02)"""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Generator
輸入：noise vector z（維度 z_dim=100）  
輸出：64×64 RGB 圖片

In [ ]:
class Generator(nn.Module):
    def __init__(self, in_dim, feature_dim=64):
        super().__init__()

        # 線性層：將 noise 向量映射到 4×4 特徵圖
        self.l1 = nn.Sequential(
            nn.Linear(in_dim, feature_dim * 8 * 4 * 4, bias=False),
            nn.BatchNorm1d(feature_dim * 8 * 4 * 4),
            nn.ReLU(),
        )

        # 反卷積層：逐步上採樣 4×4 → 8×8 → 16×16 → 32×32 → 64×64
        self.l2 = nn.Sequential(
            self._block(feature_dim * 8, feature_dim * 4),   # 4→8
            self._block(feature_dim * 4, feature_dim * 2),   # 8→16
            self._block(feature_dim * 2, feature_dim),        # 16→32
            nn.ConvTranspose2d(feature_dim, 3, kernel_size=5, stride=2,
                               padding=2, output_padding=1, bias=False),
            nn.Tanh(),  # 輸出範圍 [-1, 1]，與訓練資料正規化一致
        )

        self.apply(weights_init)

    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=5,
                               stride=2, padding=2, output_padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        y = self.l1(x)
        y = y.view(y.size(0), -1, 4, 4)  # reshape 成 4D tensor
        y = self.l2(y)
        return y

### Critic（WGAN-GP 不叫 Discriminator，因為沒有 Sigmoid）
輸入：64×64 RGB 圖片  
輸出：純量分數（非機率，所以不加 Sigmoid）

In [ ]:
class Critic(nn.Module):
    def __init__(self, in_dim=3, feature_dim=64):
        super().__init__()

        # 逐步下採樣 64×64 → 32×32 → 16×16 → 8×8 → 4×4 → 1×1
        self.l1 = nn.Sequential(
            # 第一層不加 BatchNorm（WGAN-GP 建議）
            nn.Conv2d(in_dim, feature_dim, kernel_size=5, stride=2,
                      padding=2, bias=False),
            nn.LeakyReLU(0.2),

            self._block(feature_dim,     feature_dim * 2),
            self._block(feature_dim * 2, feature_dim * 4),
            self._block(feature_dim * 4, feature_dim * 8),

            # 輸出純量分數，不加 Sigmoid
            nn.Conv2d(feature_dim * 8, 1, kernel_size=4, stride=1,
                      padding=0, bias=False),
        )

        self.apply(weights_init)

    def _block(self, in_channels, out_channels):
        # WGAN-GP 使用 InstanceNorm 而非 BatchNorm（避免 batch 統計影響 gradient penalty）
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, stride=2,
                      padding=2, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.LeakyReLU(0.2),
        )

    def forward(self, x):
        y = self.l1(x)
        y = y.view(-1)  # 壓平成 1D
        return y

## Gradient Penalty（WGAN-GP 核心）

在真實圖片與生成圖片之間做插值，強制 Critic 的梯度範數接近 1（Lipschitz 約束）。

In [ ]:
def compute_gradient_penalty(critic, real_imgs, fake_imgs, device):
    batch_size = real_imgs.size(0)

    # 在真實與假圖之間隨機插值
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)

    # 計算 Critic 對插值圖的輸出
    critic_interp = critic(interpolated)

    # 計算梯度
    gradients = torch.autograd.grad(
        outputs=critic_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interp),
        create_graph=True,
        retain_graph=True,
    )[0]

    # 梯度範數應接近 1，偏差越大懲罰越重
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    gradient_penalty = ((gradient_norm - 1) ** 2).mean()
    return gradient_penalty

## 訓練設定

In [ ]:
# 超參數
batch_size   = 64
z_dim        = 100    # noise vector 維度
lr           = 1e-4
n_epoch      = 100    # 訓練輪數（越多效果越好，建議至少 50）
n_critic     = 5     # 每更新 1 次 Generator，Critic 更新 5 次
lambda_gp    = 10    # Gradient Penalty 的權重

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用裝置：{device}')

# 建立資料載入器
dataloader = DataLoader(
    dataset, batch_size=batch_size,
    shuffle=True, num_workers=2
)

# 初始化模型
G = Generator(in_dim=z_dim).to(device)
C = Critic().to(device)

# WGAN-GP 使用 Adam(betas=(0.0, 0.9))
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
opt_C = optim.Adam(C.parameters(), lr=lr, betas=(0.0, 0.9))

# 建立輸出目錄
os.makedirs('logs', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# 固定一組 noise 用於每 epoch 的視覺化，方便追蹤訓練進度
fixed_noise = torch.randn(100, z_dim, device=device)

## 訓練迴圈

In [ ]:
G.train()
C.train()

for epoch in range(n_epoch):
    progress_bar = qqdm(dataloader)

    for i, real_imgs in enumerate(progress_bar):
        real_imgs = real_imgs.to(device)
        current_batch = real_imgs.size(0)

        # ─── 訓練 Critic ───────────────────────────────────────────
        # WGAN-GP：每個 batch 訓練 Critic n_critic 次
        for _ in range(n_critic):
            z = torch.randn(current_batch, z_dim, device=device)
            fake_imgs = G(z).detach()  # detach 避免 Generator 梯度被計算

            # Wasserstein loss：希望真圖分數高、假圖分數低
            loss_real = -C(real_imgs).mean()
            loss_fake =  C(fake_imgs).mean()

            # Gradient Penalty
            gp = compute_gradient_penalty(C, real_imgs, fake_imgs, device)

            loss_C = loss_real + loss_fake + lambda_gp * gp

            opt_C.zero_grad()
            loss_C.backward()
            opt_C.step()

        # ─── 訓練 Generator ────────────────────────────────────────
        z = torch.randn(current_batch, z_dim, device=device)
        fake_imgs = G(z)

        # 希望 Critic 對假圖給出高分（讓 Critic 無法分辨）
        loss_G = -C(fake_imgs).mean()

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        progress_bar.set_infos({
            'Loss_G': round(loss_G.item(), 4),
            'Loss_C': round(loss_C.item(), 4),
            'Epoch':  epoch + 1,
            'Step':   i + 1,
        })

    # 每個 epoch 結束，用 fixed_noise 生成樣本圖並儲存
    G.eval()
    with torch.no_grad():
        fake_samples = G(fixed_noise)
        # 反正規化：[-1,1] → [0,1]
        fake_samples = (fake_samples + 1) / 2
        grid = torchvision.utils.make_grid(fake_samples[:32], nrow=8, normalize=False)
        torchvision.utils.save_image(grid, f'logs/epoch_{epoch+1:04d}.jpg')
    G.train()

    # 每 10 個 epoch 儲存 checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f'checkpoints/G_epoch_{epoch+1}.pth')
        torch.save(C.state_dict(), f'checkpoints/C_epoch_{epoch+1}.pth')
        print(f'[Epoch {epoch+1}] checkpoint 已儲存')

print('訓練完成！')

## 訓練過程視覺化

In [ ]:
# 顯示最後一個 epoch 的生成樣本
from PIL import Image

img_path = f'logs/epoch_{n_epoch:04d}.jpg'
img = Image.open(img_path)
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f'Generated Images at Epoch {n_epoch}')
plt.show()

## Inference：生成 1000 張圖片並打包

In [ ]:
import glob

def generate_images(generator, z_dim, n_output, output_dir, device):
    os.makedirs(output_dir, exist_ok=True)
    generator.eval()

    # 分批生成，每批 100 張，避免 OOM
    batch = 100
    img_idx = 1
    with torch.no_grad():
        for _ in range(n_output // batch):
            z = torch.randn(batch, z_dim, device=device)
            imgs = generator(z)
            # 反正規化到 [0, 1]
            imgs = (imgs + 1) / 2
            for img in imgs:
                torchvision.utils.save_image(img, f'{output_dir}/{img_idx}.jpg')
                img_idx += 1

    print(f'共生成 {img_idx - 1} 張圖片至 {output_dir}/')


# 載入最佳 checkpoint（可手動選擇 epoch）
G_infer = Generator(in_dim=z_dim).to(device)
G_infer.load_state_dict(torch.load(f'checkpoints/G_epoch_{n_epoch}.pth'))

generate_images(G_infer, z_dim, n_output=1000, output_dir='generated', device=device)

In [ ]:
# 驗證生成圖片數量
generated_files = glob.glob('generated/*.jpg')
print(f'generated/ 資料夾中共有 {len(generated_files)} 張圖片')

In [ ]:
# 打包成 images.tgz（JudgeBoi 要求格式）
# 注意：tar 要在 generated/ 目錄內執行，確保 .tgz 不包含資料夾層
!cd generated && tar -zcvf ../images.tgz *.jpg
print('打包完成：images.tgz')

In [ ]:
# 確認 .tgz 檔案大小（需小於 2MB）
size_mb = os.path.getsize('images.tgz') / (1024 * 1024)
print(f'images.tgz 大小：{size_mb:.2f} MB')
if size_mb > 2:
    print('警告：檔案超過 2MB，JudgeBoi 無法接受，請降低圖片品質或壓縮率')
else:
    print('檔案大小符合要求，可上傳至 JudgeBoi')

## 顯示生成結果（抽樣展示）

In [ ]:
# 顯示前 32 張生成圖片
sample_files = sorted(glob.glob('generated/*.jpg'))[:32]
sample_imgs = [torchvision.io.read_image(f).float() / 255.0 for f in sample_files]
sample_imgs = torch.stack(sample_imgs)

grid = torchvision.utils.make_grid(sample_imgs, nrow=8, normalize=False)
plt.figure(figsize=(16, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('Generated Anime Faces (WGAN-GP)')
plt.show()